# Pro-Code: Custom Tools, Tracing & Evaluation (Part 3)

In Parts 1 and 2, you built **foundry-expert** in the Foundry portal — a no-code agent with knowledge tools and Bing search. That's great for prototyping, but production agents need more: custom runtimes, programmatic tracing, and systematic evaluation.

In this notebook, you'll do everything in Python:

- **Custom Code Interpreter** — an MCP server with your SDK packages pre-installed
- **Foundry MCP Server** — give your agent access to its own project resources
- **Tracing** — instrument your agent with OpenTelemetry for full observability
- **Evaluation** — systematically test quality and safety with built-in evaluators

### Prerequisites

- Completed [Part 1](foundry-agent-part-1.ipynb) and [Part 2](foundry-agent-part-2.ipynb)
- Python 3.10+
- An Azure subscription with a Foundry project
- A deployed model (e.g., `gpt-4o` or `gpt-5-mini`)

---

## Set Up Your Environment

Now we're in code. No more portal clicks — everything runs from this notebook.

Create a `.env` file in your project root with the following values:

| Variable | Description |
|---|---|
| `PROJECT_ENDPOINT` | Your Foundry project endpoint (e.g., `https://<resource>.services.ai.azure.com/api/projects/<project>`) |
| `MODEL_NAME` | The deployed model to use (e.g., `gpt-5-mini`) |
| `GITHUB_CONNECTION` | Project connection ID for GitHub PAT authentication |
| `CODE_INTERPRETER_URL` | URL of your custom Code Interpreter MCP server on Azure Container Apps |
| `CODE_INTERPRETER_CONNECTION` | Project connection ID for the Code Interpreter server |

In [ ]:
%pip install azure-ai-projects --pre azure-identity python-dotenv openai --quiet

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=credential,
)
openai_client = project_client.get_openai_client()
model = os.environ["MODEL_NAME"]

print(f"✅ Connected to project: {os.environ['PROJECT_ENDPOINT']}")
print(f"✅ Using model: {model}")

---

## Custom Code Interpreter

The built-in Code Interpreter runs in a sandbox — great for math and data analysis, but it doesn't have your SDK packages pre-installed. If your agent needs to write and execute real `azure-ai-projects` code, it needs a custom runtime.

The **custom code interpreter** is an MCP server running on Azure Container Apps with your packages baked in. You deploy it once (using the Bicep template from [foundry-samples](https://github.com/azure-samples/foundry-samples)), and your agent can write and execute Python against your actual project.

> **One-time setup:** Deploy the custom code interpreter container using the Bicep template from the foundry-samples repo. You'll get a `CODE_INTERPRETER_URL` and `CODE_INTERPRETER_CONNECTION` to add to your `.env` file.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

code_interpreter_agent = project_client.agents.create_version(
    agent_name="foundry-expert-code",
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a Foundry Expert that can write and execute Python code. "
            "Your runtime has azure-ai-projects, pandas, matplotlib, and other "
            "common data science packages pre-installed. When asked to demonstrate "
            "SDK usage, write working code and run it. Show the output."
        ),
        tools=[
            MCPTool(
                server_label="custom-code-interpreter",
                server_url=os.environ["CODE_INTERPRETER_URL"],
                project_connection_id=os.environ.get("CODE_INTERPRETER_CONNECTION"),
                allowed_tools=["execute"],
            ),
        ],
    ),
)
print(f"✅ Agent created: {code_interpreter_agent.name} v{code_interpreter_agent.version}")

In [ ]:
# Ask the agent to write and execute real SDK code
response = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert-code", "type": "agent_reference"}},
    input=(
        "Write a Python script using azure-ai-projects that lists all model "
        "deployments in my project, showing the model name, publisher, and SKU "
        "for each. Run it and show the output."
    ),
)

print(response.output_text)

---

## Foundry MCP Server

The Foundry MCP server gives your agent access to your **project itself** — it can list deployments, recommend models, and even deploy new models. The server URL is just your project endpoint with `/mcp_tools` appended:

```
{PROJECT_ENDPOINT}/mcp_tools?api-version=2025-05-15-preview
```

Because the agent can modify project resources, we set `require_approval="always"`. Every tool call goes through an approval flow where you explicitly approve or reject it.

In [ ]:
foundry_mcp_agent = project_client.agents.create_version(
    agent_name="foundry-expert-mcp",
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a Foundry Expert with direct access to the user's Foundry project. "
            "You can list model deployments, check capabilities, and make recommendations "
            "based on what's actually available. When recommending models, consider the "
            "user's requirements for cost, speed, and capability."
        ),
        tools=[
            MCPTool(
                server_label="foundry-project",
                server_url=f"{os.environ['PROJECT_ENDPOINT']}/mcp_tools?api-version=2025-05-15-preview",
                require_approval="always",
            ),
        ],
    ),
)
print(f"✅ Agent created: {foundry_mcp_agent.name} v{foundry_mcp_agent.version}")

In [ ]:
from openai.types.responses.response_input_param import McpApprovalResponse

# Ask the agent to recommend a model based on real deployments
response = openai_client.responses.create(
    extra_body={"agent": {"name": foundry_mcp_agent.name, "type": "agent_reference"}},
    input=(
        "What models are deployed in my project? "
        "Which would you recommend for a coding task?"
    ),
)

# Handle MCP approval requests
input_items = []
for item in response.output:
    if item.type == "mcp_approval_request":
        print(f"🔐 Tool: {item.name} on {item.server_label} | Approving...")
        input_items.append(
            McpApprovalResponse(
                type="mcp_approval_response",
                approval_request_id=item.id,
                approve=True,
            )
        )

# Send approvals and get the final response
if input_items:
    response = openai_client.responses.create(
        input=input_items,
        previous_response_id=response.id,
        extra_body={"agent": {"name": foundry_mcp_agent.name, "type": "agent_reference"}},
    )

print(response.output_text)

---

## Enable Tracing

In the portal, traces are built-in — you saw them in Part 1's Debug view. In code, you get the same observability by instrumenting with OpenTelemetry.

The `AIAgentsInstrumentor` hooks into every agent call and sends spans to Azure Monitor. Once enabled, you can view traces in the Foundry portal under **Operate > Traces** — the same rich view you saw in Part 1, but now for your pro-code agents.

> **Tip:** Enable tracing early. When something goes wrong in production, traces are the first place you'll look.

In [ ]:
%pip install azure-monitor-opentelemetry opentelemetry-sdk --quiet

In [ ]:
from azure.ai.agents.telemetry import AIAgentsInstrumentor
from azure.monitor.opentelemetry import configure_azure_monitor

# Get the Application Insights connection string from your project
app_insights_connection = project_client.telemetry.get_application_insights_connection_string()

if app_insights_connection:
    configure_azure_monitor(connection_string=app_insights_connection)
    print("✅ Azure Monitor configured")
else:
    print("⚠️ No Application Insights configured — add it in the Foundry portal under Settings")

# Instrument the agents SDK
AIAgentsInstrumentor().instrument()

# Enable content recording so traces include inputs/outputs
os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"

print("✅ Tracing enabled — view traces in the Foundry portal under Operate > Traces")

---

## Agent Evaluation

Your agent works in the playground. It works in code. But does it work *reliably*? Evaluation is how you move from "it seems fine" to "I have metrics that prove it."

Foundry provides **cloud-based evaluation** that runs your agent against test queries and scores the results using built-in evaluators:

| Evaluator | What it measures |
|---|---|
| `builtin.violence` | Safety — detects violent content |
| `builtin.fluency` | Quality — is the response well-written? |
| `builtin.task_adherence` | Quality — did the agent follow instructions? |

You define test cases, create an evaluation, run it against your agent, and get scored results — all from code.

In [ ]:
from typing import Union
from openai.types.evals import RunCreateResponse, RunRetrieveResponse
from azure.ai.projects.models import DataSourceConfigCustom
import time

# Define the data schema for evaluation
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
    include_sample_schema=True,
)

# Define testing criteria with built-in evaluators
testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "violence_detection",
        "evaluator_name": "builtin.violence",
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": model},
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "task_adherence",
        "evaluator_name": "builtin.task_adherence",
        "initialization_parameters": {"deployment_name": model},
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_items}}"},
    },
]

# Create the evaluation object
eval_object = openai_client.evals.create(
    name="Foundry Expert - Quality & Safety",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print(f"✅ Evaluation created: {eval_object.name} (id: {eval_object.id})")

# Define test queries and run them against the Foundry MCP agent
data_source = {
    "type": "azure_ai_target_completions",
    "source": {
        "type": "file_content",
        "content": [
            {"item": {"query": "What models are available in my project?"}},
            {"item": {"query": "How do I create an agent with the Python SDK?"}},
            {"item": {"query": "What is the difference between a Foundry resource and a project?"}},
        ],
    },
    "input_messages": {
        "type": "template",
        "template": [
            {"type": "message", "role": "user", "content": {"type": "input_text", "text": "{{item.query}}"}}
        ],
    },
    "target": {
        "type": "azure_ai_agent",
        "name": foundry_mcp_agent.name,
        "version": foundry_mcp_agent.version,
    },
}

# Submit the evaluation run
eval_run: Union[RunCreateResponse, RunRetrieveResponse] = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f"Eval run for {foundry_mcp_agent.name}",
    data_source=data_source,
)
print(f"🚀 Evaluation run submitted (id: {eval_run.id})")

# Poll until the evaluation run completes
while eval_run.status in ("queued", "in_progress", "running"):
    time.sleep(5)
    eval_run = openai_client.evals.runs.retrieve(eval_id=eval_object.id, run_id=eval_run.id)
    print(f"   Status: {eval_run.status}")

print(f"\n✅ Evaluation complete — status: {eval_run.status}")
print(f"   Result counts: {eval_run.result_counts}")
print(f"\n📊 View detailed results in the Foundry portal under Assess > Evaluations")

---

## Clean Up

In [ ]:
agents_to_clean = [
    ("foundry-expert-code", code_interpreter_agent.version),
    ("foundry-expert-mcp", foundry_mcp_agent.version),
]

for name, version in agents_to_clean:
    try:
        project_client.agents.delete_version(agent_name=name, version=version)
        print(f"🗑️ Deleted: {name} v{version}")
    except Exception as e:
        print(f"⚠️ Could not delete {name} v{version}: {e}")

# Clean up the evaluation
try:
    openai_client.evals.delete(eval_id=eval_object.id)
    print(f"🗑️ Deleted evaluation: {eval_object.id}")
except Exception as e:
    print(f"⚠️ Could not delete evaluation: {e}")

print("\n✅ Cleanup complete")

---

## What's Next

You've gone fully pro-code — custom code interpreter, Foundry MCP server, tracing, and evaluations. Your agent can execute real SDK code and manage its own project resources.

In **Part 4**, you'll take this to production:
- Red team your agent to find vulnerabilities
- Deploy with `azd up` to Azure Container Apps
- Publish to Microsoft 365 Copilot and Teams
- Monitor with production traces and continuous evaluations

👉 [Continue to Part 4: Red Team, Deploy & Optimize](foundry-agent-part-4.ipynb)